# Offline FNN Inference from VEstim Job Folder

This notebook runs **offline inference** outside the GUI using artifacts saved in a VEstim job folder.

It is intentionally **FNN-only** and reuses core repo services for parity with in-tool testing logic.


In [21]:
import os
import json
import datetime
import traceback
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from vestim.services.data_processor.src.data_augment_service import DataAugmentService
from vestim.services.model_testing.src.continuous_testing_service import ContinuousTestingService
from vestim.services.data_processor.src import normalization_service as norm_svc

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)


In [22]:
# ===== USER CONFIG =====
JOB_FOLDER = r"C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final"
TEST_DIR = r"C:\Users\dehuryb\Downloads\SD60_results\SD60_results\test"
TEST_GLOB = ('*.csv', '*.xlsx', '*.xls')

# Optional: force CPU for reproducibility across machines
FORCE_CPU = False

# Optional: inference smoothing override. Keep {} to use task-saved values.
INFERENCE_FILTER_OVERRIDE = {}

job_folder = Path(JOB_FOLDER)
test_dir = Path(TEST_DIR)
assert job_folder.exists(), f'Job folder not found: {job_folder}'
assert test_dir.exists(), f'Test dir not found: {test_dir}'
print('JOB_FOLDER =', job_folder)
print('TEST_DIR    =', test_dir)


JOB_FOLDER = C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final
TEST_DIR    = C:\Users\dehuryb\Downloads\SD60_results\SD60_results\test


In [23]:
def _log(msg):
    print(msg)

def scan_test_files(test_dir: Path, patterns=('*.csv', '*.xlsx', '*.xls')):
    files = []
    for p in patterns:
        files.extend(sorted(test_dir.glob(p)))
    return files

def read_input_file(path: Path) -> pd.DataFrame:
    ext = path.suffix.lower()
    if ext == '.csv':
        return pd.read_csv(path)
    if ext in ('.xlsx', '.xls'):
        return pd.read_excel(path, sheet_name=0)
    raise ValueError(f'Unsupported file type: {path.suffix}')

def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def apply_automatic_augmentation(df: pd.DataFrame, aug_metadata: dict, augment_service: DataAugmentService) -> pd.DataFrame:
    result_df = df.copy()
    padding_info = aug_metadata.get('padding', {})

    applied_filters = aug_metadata.get('applied_filters', [])
    temp_filter_padding_length = 0
    if applied_filters and padding_info.get('applied', False):
        padding_mode = padding_info.get('mode')
        removed_before_save = padding_info.get('removed_before_save', False)
        if padding_mode == 'temporary_pre_filter' or removed_before_save:
            temp_filter_padding_length = int(padding_info.get('length', 0) or 0)

    if applied_filters and temp_filter_padding_length > 0:
        _log(f'Applying temporary pre-filter padding (length={temp_filter_padding_length})')
        result_df = augment_service.pad_data(
            result_df,
            temp_filter_padding_length,
            resample_freq_for_time_padding=padding_info.get('resampling_frequency_for_padding')
        )

    for filt in applied_filters:
        col = filt['column']
        if col not in result_df.columns:
            raise ValueError(f"Missing required filter column '{col}' in test data")
        _log(f"Applying Butterworth filter on '{col}' -> '{filt.get('output_column_name', '')}'")
        result_df = augment_service.apply_butterworth_filter(
            result_df,
            column_name=col,
            corner_frequency=filt['corner_frequency'],
            sampling_rate=filt['sampling_rate'],
            filter_order=filt['filter_order'],
            output_column_name=filt['output_column_name']
        )

    if applied_filters and temp_filter_padding_length > 0:
        _log(f'Removing temporary filter padding (length={temp_filter_padding_length})')
        result_df = augment_service.remove_padding(result_df, temp_filter_padding_length)

    created_columns = aug_metadata.get('created_columns', [])
    if created_columns:
        formulas = [(x['column_name'], x['formula']) for x in created_columns]
        _log(f'Creating {len(formulas)} calculated columns')
        result_df = augment_service.create_columns(result_df, formulas)

    resampling_info = aug_metadata.get('resampling', {})
    if resampling_info.get('applied', False):
        freq = resampling_info.get('frequency')
        source_hz = resampling_info.get('source_frequency_hz', resampling_info.get('original_sample_rate_hz'))
        _log(f'Applying resampling: {freq} (source_hz={source_hz})')
        result_df = augment_service.resample_data(result_df, freq, source_sampling_rate_hz=source_hz)

    if (not applied_filters) and padding_info.get('applied', False):
        plen = int(padding_info.get('length', 0) or 0)
        if plen > 0:
            _log(f'Applying persistent padding (length={plen})')
            result_df = augment_service.pad_data(
                result_df,
                plen,
                resample_freq_for_time_padding=padding_info.get('resampling_frequency_for_padding')
            )

    return result_df

def apply_normalization_if_needed(df: pd.DataFrame, job_metadata: dict, scaler_path: Path) -> pd.DataFrame:
    out = df.copy()
    if not job_metadata.get('normalization_applied', False):
        return out

    scaler = norm_svc.load_scaler(str(scaler_path))
    if scaler is None:
        raise RuntimeError(f'Could not load scaler from {scaler_path}')

    scaler_features = list(scaler.feature_names_in_) if hasattr(scaler, 'feature_names_in_') else []
    if not scaler_features:
        normalized_columns = job_metadata.get('normalized_columns', [])
        scaler_features = [c for c in normalized_columns if c in out.columns]

    missing = [c for c in scaler_features if c not in out.columns]
    if missing:
        raise ValueError(f'Missing scaler-required columns: {missing}')

    # Match standalone manager preprocessing for object/time-like columns before scaler.transform()
    mm_ss_pattern = re.compile(r'^\d+:\d+(?:\.\d+)?$')
    hh_mm_ss_pattern = re.compile(r'^\d+:\d{2}:\d{2}(?:\.\d+)?$')

    for col in scaler_features:
        if col not in out.columns:
            continue

        col_dtype_str = str(out[col].dtype)
        if col_dtype_str not in ('object', 'string', 'str'):
            continue

        col_as_str = out[col].astype(str).str.strip()
        non_empty = col_as_str[col_as_str != '']
        sample_val = non_empty.iloc[0] if len(non_empty) > 0 else ''

        # MM:SS(.s) duration (e.g. 56:43.4)
        if sample_val and mm_ss_pattern.match(sample_val):
            def mm_ss_to_seconds(x):
                if pd.isna(x):
                    return np.nan
                value = str(x).strip()
                if value == '':
                    return np.nan
                if mm_ss_pattern.match(value):
                    parts = value.split(':')
                    return int(parts[0]) * 60 + float(parts[1])
                numeric = pd.to_numeric(value, errors='coerce')
                return numeric if not pd.isna(numeric) else np.nan

            out[col] = out[col].apply(mm_ss_to_seconds)

        # HH:MM:SS(.s) duration
        elif sample_val and hh_mm_ss_pattern.match(sample_val):
            def hh_mm_ss_to_seconds(x):
                if pd.isna(x):
                    return np.nan
                value = str(x).strip()
                if value == '':
                    return np.nan
                if hh_mm_ss_pattern.match(value):
                    parts = value.split(':')
                    return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
                numeric = pd.to_numeric(value, errors='coerce')
                return numeric if not pd.isna(numeric) else np.nan

            out[col] = out[col].apply(hh_mm_ss_to_seconds)

        # datetime-like strings (convert to elapsed seconds from first timestamp)
        elif sample_val and any(ch in sample_val for ch in ('-', '/')) and ':' in sample_val and ' ' in sample_val:
            parsed_dt = pd.to_datetime(out[col], errors='coerce')
            if parsed_dt.notna().any():
                first_ts = parsed_dt.dropna().iloc[0]
                out[col] = (parsed_dt - first_ts).dt.total_seconds()
            else:
                out[col] = pd.to_numeric(out[col], errors='coerce')

        else:
            out[col] = pd.to_numeric(out[col], errors='coerce')

    out[scaler_features] = scaler.transform(out[scaler_features])
    return out

def compute_units(target_column: str):
    t = (target_column or '').lower()
    if 'voltage' in t:
        return 'mV', 1000.0
    if 'soc' in t:
        return '%SOC', 100.0
    if 'temperature' in t or 'temp' in t:
        return '°C', 1.0
    return '', 1.0

def scan_fnn_tasks(models_dir: Path):
    tasks = []
    for arch_dir in sorted(models_dir.iterdir()):
        if not arch_dir.is_dir():
            continue
        for task_dir in sorted(arch_dir.iterdir()):
            if not task_dir.is_dir():
                continue
            task_info_file = task_dir / 'task_info.json'
            model_file = task_dir / 'best_model.pth'
            if not (task_info_file.exists() and model_file.exists()):
                continue
            task_info = load_json(task_info_file)
            model_type = str(task_info.get('model_type', task_info.get('model_metadata', {}).get('model_type', ''))).upper()
            if model_type != 'FNN':
                continue
            task_info['architecture_name'] = arch_dir.name
            task_info['task_name'] = task_dir.name
            task_info['task_path'] = str(task_dir)
            task_info['model_file'] = str(model_file)
            tasks.append(task_info)
    return tasks


In [24]:
job_metadata_path = job_folder / 'job_metadata.json'
aug_metadata_path = job_folder / 'augmentation_metadata.json'
scaler_path = job_folder / 'scalers' / 'augmentation_scaler.joblib'
models_dir = job_folder / 'models'

assert job_metadata_path.exists(), f'Missing: {job_metadata_path}'
assert models_dir.exists(), f'Missing: {models_dir}'

job_metadata = load_json(job_metadata_path)
aug_metadata = load_json(aug_metadata_path) if aug_metadata_path.exists() else None

test_files = scan_test_files(test_dir, TEST_GLOB)
assert test_files, f'No test files found in {test_dir}'

fnn_tasks = scan_fnn_tasks(models_dir)
assert fnn_tasks, f'No FNN task models found under {models_dir}'

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
offline_dir = job_folder / f'offline_testing_results_{timestamp}'
offline_dir.mkdir(parents=True, exist_ok=True)

print('Found test files:', len(test_files))
for f in test_files:
    print('  -', f.name)
print('Found FNN tasks:', len(fnn_tasks))
for t in fnn_tasks:
    print('  -', t['architecture_name'], '/', t['task_name'])
print('Output dir:', offline_dir)


Found test files: 24
  - HWFET_0_degC.csv
  - HWFET_10_degC.csv
  - HWFET_25_degC.csv
  - HWFET_40_degC.csv
  - HWFET_n10_degC.csv
  - HWFET_n20_degC.csv
  - LA92_0_degC.csv
  - LA92_10_degC.csv
  - LA92_25_degC.csv
  - LA92_40_degC.csv
  - LA92_n10_degC.csv
  - LA92_n20_degC.csv
  - UDDS_0_degC.csv
  - UDDS_10_degC.csv
  - UDDS_25_degC.csv
  - UDDS_40_degC.csv
  - UDDS_n10_degC.csv
  - UDDS_n20_degC.csv
  - US06_0_degC.csv
  - US06_10_degC.csv
  - US06_25_degC.csv
  - US06_40_degC.csv
  - US06_n10_degC.csv
  - US06_n20_degC.csv
Found FNN tasks: 1
  - FNN_256_128 / B4096_LR_SLR_VP300_rep_4
Output dir: C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final\offline_testing_results_20260515_192043


In [25]:
device = torch.device('cpu' if FORCE_CPU else ('cuda' if torch.cuda.is_available() else 'cpu'))
cts = ContinuousTestingService(device=device)
augment_service = DataAugmentService()

summary_rows = []

for test_file in test_files:
    print('\n' + '='*90)
    print(f'PROCESSING TEST FILE: {test_file.name}')

    raw_df = read_input_file(test_file)
    prepared_df = raw_df.copy()

    if aug_metadata is not None:
        prepared_df = apply_automatic_augmentation(prepared_df, aug_metadata, augment_service)

    if job_metadata.get('normalization_applied', False):
        prepared_df = apply_normalization_if_needed(prepared_df, job_metadata, scaler_path)

    # Use a temporary normalized file for ContinuousTestingService input (do not persist prepared files)
    with tempfile.NamedTemporaryFile(prefix='vestim_prepared_', suffix='.csv', delete=False) as tf:
        prepared_file = Path(tf.name)
    prepared_df.to_csv(prepared_file, index=False)

    for task in fnn_tasks:
        arch = task['architecture_name']
        task_name = task['task_name']
        model_file = task['model_file']

        print(f'  -> Testing {arch}/{task_name}')

        task_payload = dict(task)
        task_payload['job_metadata'] = job_metadata
        task_payload['job_folder_augmented_from'] = str(job_folder)
        task_payload['hyperparams'] = dict(task_payload.get('hyperparams', {}))
        if INFERENCE_FILTER_OVERRIDE:
            task_payload['hyperparams'].update(INFERENCE_FILTER_OVERRIDE)

        hp = task_payload.get('hyperparams', {})
        data_cfg = task_payload.get('data_config', {})
        feature_columns = hp.get('FEATURE_COLUMNS', data_cfg.get('feature_columns', []))
        target_column = hp.get('TARGET_COLUMN', data_cfg.get('target_column'))
        lookback_val = hp.get('LOOKBACK', data_cfg.get('lookback', 0))
        lookback = 0 if lookback_val in ('N/A', None) else int(lookback_val)

        task_payload['data_loader_params'] = task_payload.get('data_loader_params') or {
            'feature_columns': feature_columns,
            'target_column': target_column
        }

        if feature_columns:
            missing = [c for c in feature_columns if c not in prepared_df.columns]
            if missing:
                raise ValueError(f'Missing required feature columns for {arch}/{task_name}: {missing}')

        result = cts.run_continuous_testing(
            task=task_payload,
            model_path=model_file,
            test_file_path=str(prepared_file),
            is_first_file=True,
            warmup_samples=lookback
        )

        if result is None:
            raise RuntimeError(f'ContinuousTestingService returned None for {arch}/{task_name}')

        preds = np.ravel(np.asarray(result.get('predictions', [])))
        truev = np.ravel(np.asarray(result.get('true_values', [])))
        n = min(len(preds), len(truev))
        preds = preds[-n:]
        truev = truev[-n:]

        if n == 0:
            mae = rmse = r2 = maxe = np.nan
        else:
            err = truev - preds
            mae = float(np.mean(np.abs(err)))
            mse = float(np.mean(err**2))
            rmse = float(np.sqrt(mse))
            denom = float(np.sum((truev - np.mean(truev))**2))
            r2 = float(1 - (np.sum(err**2) / denom)) if denom > 0 else np.nan
            maxe = float(np.max(np.abs(err)))

        unit_name, factor = compute_units(target_column or '')
        rmse_disp = rmse * factor if pd.notna(rmse) else np.nan
        maxe_disp = maxe * factor if pd.notna(maxe) else np.nan
        mae_disp = mae * factor if pd.notna(mae) else np.nan

        ts_n = min(n, len(prepared_df))
        ts_series = prepared_df['Timestamp'].to_numpy()[-ts_n:] if ('Timestamp' in prepared_df.columns) else np.arange(ts_n)
        if 'Time (h)' in prepared_df.columns:
            time_h_series = pd.to_numeric(prepared_df['Time (h)'], errors='coerce').to_numpy()[-ts_n:]
        else:
            time_h_series = np.arange(ts_n, dtype=float) / 3600.0

        pred_df = pd.DataFrame({
            'Timestamp': ts_series,
            'Time (h)': time_h_series,
            f'True_{target_column}': truev,
            f'Predicted_{target_column}': preds,
            f'Error ({unit_name})': (truev - preds) * factor
        })

        model_out_dir = offline_dir / arch / task_name
        model_out_dir.mkdir(parents=True, exist_ok=True)
        out_name = f"{test_file.stem}_predictions.csv"
        out_path = model_out_dir / out_name
        pred_df.to_csv(out_path, index=False)

        summary_rows.append({
            'test_file': test_file.name,
            'architecture': arch,
            'task': task_name,
            'target_column': target_column,
            'unit': unit_name,
            'samples_used': int(n),
            'RMSE_display': rmse_disp,
            'MAXE_display': maxe_disp,
            'MAE_display': mae_disp,
            'R2': r2,
            'RMSE_unconverted': rmse,
            'MAXE_unconverted': maxe,
            'MAE_unconverted': mae,
            'predictions_file': str(out_path)
        })

    try:
        prepared_file.unlink(missing_ok=True)
    except Exception:
        pass

summary_df = pd.DataFrame(summary_rows)
summary_path = offline_dir / f'offline_testing_summary_{timestamp}.csv'
summary_df.to_csv(summary_path, index=False)

print('\nDONE')
print('Summary:', summary_path)
print('Rows   :', len(summary_df))
display(summary_df.head(20))


ContinuousTestingService initialized with device: cpu (type: <class 'torch.device'>)

PROCESSING TEST FILE: HWFET_0_degC.csv
Applying Butterworth filter on 'Watts' -> 'P_filtr_0p002'
Applying Butterworth filter on 'Watts' -> 'P_filter_0p02'
Scaler loaded from C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final\scalers\augmentation_scaler.joblib


C:\Users\dehuryb\.python_matlab\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  -> Testing FNN_256_128/B4096_LR_SLR_VP300_rep_4
Running continuous testing for: C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_zcj2lv_r.csv
DEBUG: Loading model with device: cpu
Created FNN model instance
DEBUG: Model device after creation: cpu
Model loaded: C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final\models\FNN_256_128\B4096_LR_SLR_VP300_rep_4\best_model.pth
DEBUG: Model moved to device: cpu
DEBUG: First model parameter device: cpu
No hidden states needed for FNN model
Loaded scaler from C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final\scalers\augmentation_scaler.joblib
DEBUG - Scaler details:
  Type: <class 'sklearn.preprocessing._data.MinMaxScaler'>
  Feature names: None
Processing 31547 samples from C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_zcj2lv_r.csv
Normalization applied during training: True
Loading processed te

C:\Users\dehuryb\.python_matlab\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Generated 31547 predictions (after warmup: 0)
Data processing mode: processed file (from data augmentation pipeline)
Normalization was applied during training: True
Data shapes - Predictions: (31547,), True values: (31547,)
Sample values - Pred: [1. 1. 1.], True: [0.9978643 0.9977954 0.9976576]
DEBUG - Before denormalization:
  Raw predictions (first 5): [1. 1. 1. 1. 1.]
  Raw true values (first 5): [0.9978643 0.9977954 0.9976576 0.9976576 0.9975198]
  Prediction range: [0.249410, 1.000000]
  True values range: [0.210127, 1.000000]
Normalization was applied during training - denormalizing predictions and true values
Denormalizing values for Voltage using standardized method
Scaler type: <class 'sklearn.preprocessing._data.MinMaxScaler'>
Normalized columns: ['Prog Time', 'Step Time', 'Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Watts', 'TSW', 'Cnt', 'SOC', 'P_filtr_0p002', 'P_filter_0p02']
Sample normalized values - Pred: [1. 1. 1.], True: [0.9978643 0.9977954 0.9976576]


...

C:\Users\dehuryb\.python_matlab\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  -> Testing FNN_256_128/B4096_LR_SLR_VP300_rep_4
Running continuous testing for: C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_1nbctxol.csv
DEBUG: Loading model with device: cpu
Created FNN model instance
DEBUG: Model device after creation: cpu
Model loaded: C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final\models\FNN_256_128\B4096_LR_SLR_VP300_rep_4\best_model.pth
DEBUG: Model moved to device: cpu
DEBUG: First model parameter device: cpu
No hidden states needed for FNN model
Processing 27561 samples from C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_1nbctxol.csv
Normalization applied during training: True
Loading processed test file: C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_1nbctxol.csv
DEBUG - Processed test data analysis:
  Available columns: ['Temperature', 'Watts', 'SOC', 'P_filtr_0p002', 'P_filter_0p02', 'Voltage']
  Target column 'Voltage' sample values: [0.99779538 0.9976576  0.99751

C:\Users\dehuryb\.python_matlab\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


  -> Testing FNN_256_128/B4096_LR_SLR_VP300_rep_4
Running continuous testing for: C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_ligoxfel.csv
DEBUG: Loading model with device: cpu
Created FNN model instance
DEBUG: Model device after creation: cpu
Model loaded: C:\Users\dehuryb\Downloads\SD60_results\SD60_results\job_20250922-231821_2_filter_LR_50_Large_2_best_model_filtered_final\models\FNN_256_128\B4096_LR_SLR_VP300_rep_4\best_model.pth
DEBUG: Model moved to device: cpu
DEBUG: First model parameter device: cpu
No hidden states needed for FNN model
Processing 28503 samples from C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_ligoxfel.csv
Normalization applied during training: True
Loading processed test file: C:\Users\dehuryb\AppData\Local\Temp\vestim_prepared_ligoxfel.csv
DEBUG - Processed test data analysis:
  Available columns: ['Temperature', 'Watts', 'SOC', 'P_filtr_0p002', 'P_filter_0p02', 'Voltage']
  Target column 'Voltage' sample values: [0.99779538 0.9976576  0.99751

,test_file,architecture,task,target_column,unit,samples_used,RMSE_display,MAXE_display,MAE_display,R2,RMSE_unconverted,MAXE_unconverted,MAE_unconverted,predictions_file
0,HWFET_0_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,31547,8.234402,64.760836,5.631910,0.999256,0.008234,0.064761,0.005632,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
1,HWFET_10_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,30593,7.028580,38.043181,4.563897,0.999424,0.007029,0.038043,0.004564,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
2,HWFET_25_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,27394,5.727877,35.999972,3.938765,0.999557,0.005728,0.036000,0.003939,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
3,HWFET_40_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,35181,10.930235,40.899987,7.201528,0.998720,0.010930,0.040900,0.007202,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
4,HWFET_n10_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,32172,11.856523,99.551313,8.482339,0.998520,0.011857,0.099551,0.008482,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
5,HWFET_n20_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,32586,17.184857,134.010346,12.041830,0.997171,0.017185,0.134010,0.012042,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
6,LA92_0_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,42511,7.343024,90.086890,5.033374,0.999335,0.007343,0.090087,0.005033,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
7,LA92_10_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,41871,6.763906,37.700014,4.608023,0.999395,0.006764,0.037700,0.004608,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
8,LA92_25_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,39485,6.218047,43.016172,4.284959,0.999428,0.006218,0.043016,0.004285,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...
9,LA92_40_degC.csv,FNN_256_128,B4096_LR_SLR_VP300_rep_4,Voltage,mV,46523,9.301605,53.199987,6.054604,0.998956,0.009302,0.053200,0.006055,C:\Users\dehuryb\Downloads\SD60_results\SD60_r...


## Notes
- Output directory is created as `offline_testing_results_<timestamp>` under the selected job folder.
- Per-model per-test prediction CSVs are saved with `_predictions.csv` suffix.
- Summary is saved as `offline_testing_summary_<timestamp>.csv`.
- This notebook intentionally targets **FNN only** to keep parity checks focused.
